# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prarthanamahesh21-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token available:", HF_TOKEN is not None)

HF token available: True


In [18]:
!pip -q install datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token available:", HF_TOKEN is not None)

HF token available: True


In [19]:
dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train",
    token=HF_TOKEN
)

print("dim_content loaded")
print(dim_content)

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

dim_content loaded
Dataset({
    features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
    num_rows: 519606
})


In [20]:
fact_query_90d = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train",
    token=HF_TOKEN
)

print("fact_query_90d loaded")
print(fact_query_90d)

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

fact_query_90d loaded
Dataset({
    features: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share'],
    num_rows: 2414248
})


In [21]:
print("CONTENT COLUMNS:")
print(dim_content.column_names)

print("\nQUERY COLUMNS:")
print(fact_query_90d.column_names)

CONTENT COLUMNS:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

QUERY COLUMNS:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [22]:
# Convert Hugging Face datasets to pandas

content_df = dim_content.to_pandas()
query_df = fact_query_90d.to_pandas()

print("Content shape:", content_df.shape)
print("Query shape:", query_df.shape)

Content shape: (519606, 26)
Query shape: (2414248, 21)


In [23]:
# Build page-level signal frame

# Keep one content row per page
content_page = (
    content_df[
        [
            "client_hash_id",
            "content_hash_id",
            "content_created_date",
            "word_count"
        ]
    ]
    .drop_duplicates("content_hash_id")
    .copy()
)

# Aggregate query metrics to page level
query_page = (
    query_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        gsc_impressions_90d=("impressions_90d", "sum"),
        gsc_clicks_90d=("clicks_90d", "sum")
    )
)

# Merge the two sources
audit_df = content_page.merge(
    query_page,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Calculate content age as of March 31, 2026
audit_df["content_created_date"] = pd.to_datetime(
    audit_df["content_created_date"],
    errors="coerce"
)

audit_df["content_age_days"] = (
    pd.Timestamp("2026-03-31")
    - audit_df["content_created_date"]
).dt.days

# Fill missing query activity with zero
audit_df[
    ["gsc_impressions_90d", "gsc_clicks_90d"]
] = audit_df[
    ["gsc_impressions_90d", "gsc_clicks_90d"]
].fillna(0)

print("Audit frame shape:", audit_df.shape)
display(audit_df.head())

Audit frame shape: (519606, 7)


,client_hash_id,content_hash_id,content_created_date,word_count,gsc_impressions_90d,gsc_clicks_90d,content_age_days
0,client_04660893ae39614a,content_004de9653278b5a4,2026-05-30,2555.0,0.0,0.0,-60
1,client_04660893ae39614a,content_00dc5efae381b2ab,2026-06-12,2430.0,0.0,0.0,-73
2,client_04660893ae39614a,content_01410f2556c327ac,2026-05-09,2645.0,0.0,0.0,-39
3,client_04660893ae39614a,content_019f27f634053ca7,2026-06-15,2522.0,0.0,0.0,-76
4,client_04660893ae39614a,content_01efa71faea45dcc,2026-05-21,2552.0,0.0,0.0,-51


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Distribution observations

The search signals are strongly right-skewed. GSC impressions have a skewness of 40.74 and clicks have a skewness of 65.82, with medians of 0 for both signals, while a small number of pages have much larger values. Word count is less skewed (0.75), with a median of 2,593 words. Content age is comparatively balanced, although the observed minimum of -97 days indicates some dates are later than the March 31, 2026 reference date and should be treated as a data-quality issue. These distributions support using percentile-based comparisons rather than relying only on averages or simple thresholds.


In [24]:
# Section 1 — Distribution summary

signal_cols = [
    "content_age_days",
    "word_count",
    "gsc_impressions_90d",
    "gsc_clicks_90d"
]

print("Distribution summary:")

display(
    audit_df[signal_cols]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .T
)

print("\nSkewness:")

display(
    audit_df[signal_cols]
    .skew()
    .sort_values(ascending=False)
)

Distribution summary:


,count,mean,std,min,25%,50%,75%,90%,95%,max
content_age_days,519606.0,206.350263,175.963027,-97.0,49.0,217.0,330.0,487.0,494.0,531.0
word_count,341838.0,2472.052680,1087.753954,0.0,1539.0,2593.0,3038.0,3717.0,4247.0,29341.0
gsc_impressions_90d,519606.0,407.506873,4084.829939,0.0,0.0,0.0,11.0,307.0,1065.0,606884.0
gsc_clicks_90d,519606.0,0.885128,12.991604,0.0,0.0,0.0,0.0,0.0,2.0,2422.0



Skewness:


,0
gsc_clicks_90d,65.817982
gsc_impressions_90d,40.741059
word_count,0.751621
content_age_days,0.115176


In [11]:
import os

print(os.listdir("/content")[:50])

['.config', 'sample_data']


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [25]:
print("Content columns available:")
print(content_df.columns.tolist())

print("\nQuery columns available:")
print(query_df.columns.tolist())

Content columns available:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

Query columns available:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [27]:
# Check whether a label or flag exists in the currently loaded data

for name, obj in list(globals().items()):
    if hasattr(obj, "columns"):
        matches = [
            c for c in obj.columns
            if "label" in c.lower() or "flag" in c.lower()
        ]
        if matches:
            print(name, "->", matches)

### Signal test #1 — Content age

I tested whether older content tends to have lower recent search activity. The comparison uses median 90-day impressions across content-age quartiles. The result is **MIXED** if the relationship does not move consistently in one direction, **CONFIRMED** if impressions generally decline as age increases, or **OPPOSITE** if they generally increase. This is a directional test, not evidence of causation.


In [28]:
# Signal test #1 — Content age vs search impressions

age_test = (
    audit_df
    .assign(
        age_group=pd.qcut(
            audit_df["content_age_days"],
            q=4,
            duplicates="drop"
        )
    )
    .groupby("age_group", observed=True)
    .agg(
        pages=("content_hash_id", "count"),
        median_impressions=("gsc_impressions_90d", "median"),
        mean_impressions=("gsc_impressions_90d", "mean")
    )
)

print("Search activity by content-age quartile:")
display(age_test)

Search activity by content-age quartile:


,pages,median_impressions,mean_impressions
age_group,,,
"(-97.001, 49.0]",130794,0.0,477.033885
"(49.0, 217.0]",130672,0.0,622.966902
"(217.0, 330.0]",128519,0.0,271.238657
"(330.0, 531.0]",129621,0.0,255.253346


### Signal test #2 — Word count

I tested whether pages with different amounts of content show different levels of recent search activity. The comparison uses median 90-day impressions across word-count quartiles. The result is **MIXED** if there is no consistent pattern, **CONFIRMED** if higher word counts generally correspond to higher impressions, or **OPPOSITE** if the direction is generally reversed.


In [29]:
# Signal test #2 — Word count vs search impressions

word_test = (
    audit_df
    .dropna(subset=["word_count"])
    .assign(
        word_group=pd.qcut(
            audit_df["word_count"],
            q=4,
            duplicates="drop"
        )
    )
    .groupby("word_group", observed=True)
    .agg(
        pages=("content_hash_id", "count"),
        median_impressions=("gsc_impressions_90d", "median"),
        mean_impressions=("gsc_impressions_90d", "mean")
    )
)

print("Search activity by word-count quartile:")
display(word_test)

Search activity by word-count quartile:


,pages,median_impressions,mean_impressions
word_group,,,
"(-0.001, 1539.0]",85585,0.0,87.678074
"(1539.0, 2593.0]",85386,0.0,611.338803
"(2593.0, 3038.0]",85408,0.0,908.147246
"(3038.0, 29341.0]",85459,0.0,567.577411


### Signal test #3 — Search impressions and clicks

I tested whether pages with higher 90-day search impressions also tend to receive more clicks. The comparison uses impression quartiles and the median number of clicks in each group. The result is **CONFIRMED** if clicks generally increase with impressions, **OPPOSITE** if they decrease, or **MIXED** if the pattern is inconsistent.


In [30]:
# Signal test #3 — Search impressions vs clicks

impression_test = (
    audit_df
    .assign(
        impression_group=pd.qcut(
            audit_df["gsc_impressions_90d"],
            q=4,
            duplicates="drop"
        )
    )
    .groupby("impression_group", observed=True)
    .agg(
        pages=("content_hash_id", "count"),
        median_clicks=("gsc_clicks_90d", "median"),
        mean_clicks=("gsc_clicks_90d", "mean")
    )
)

print("Clicks by impression quartile:")
display(impression_test)

Clicks by impression quartile:


,pages,median_clicks,mean_clicks
impression_group,,,
"(-0.001, 11.0]",391443,0.0,0.000156
"(11.0, 606884.0]",128163,0.0,3.588064


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Flag-linked test — Content age threshold

The baseline flag uses content age as a reason to prioritize older pages for review. I tested the 180-day threshold by comparing search activity for pages below and above the threshold. The result is **MIXED** because the age-based groups do not provide a consistently stronger search-activity signal for the older pages. This suggests that age can be a useful decision-support signal, but the threshold should not be treated as proof that a page needs a refresh.


In [31]:
# Section 3 — Flag-linked test
# Baseline assumption: pages >= 180 days are more likely to need review

AGE_THRESHOLD = 180

flag_linked = (
    audit_df
    .assign(
        age_flag=np.where(
            audit_df["content_age_days"] >= AGE_THRESHOLD,
            "Age >= 180 days",
            "Age < 180 days"
        )
    )
    .groupby("age_flag")
    .agg(
        pages=("content_hash_id", "count"),
        median_impressions=("gsc_impressions_90d", "median"),
        mean_impressions=("gsc_impressions_90d", "mean"),
        median_clicks=("gsc_clicks_90d", "median"),
        mean_clicks=("gsc_clicks_90d", "mean")
    )
)

print("Flag-linked test — 180-day content-age threshold:")
display(flag_linked)

Flag-linked test — 180-day content-age threshold:


,pages,median_impressions,mean_impressions,median_clicks,mean_clicks
age_flag,,,,,
Age < 180 days,205542,0.0,609.811727,0.0,1.387395
Age >= 180 days,314064,0.0,275.106647,0.0,0.556415


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Flag-linked test — Content age threshold

The baseline flag uses content age as a reason to prioritize older pages for review. I tested the 180-day threshold by comparing search activity for pages below and above the threshold. The result is **CONFIRMED** directionally: pages aged 180 days or more have lower mean 90-day impressions (275.11 vs. 609.81) and lower mean clicks (0.56 vs. 1.39). This supports using age as a decision-support signal, but it does not show that age itself causes lower performance.


### What this means in practice

The audit suggests that content age is a useful directional signal for review prioritization, while word count has a more mixed relationship with search activity. The 180-day age flag is supported by the observed difference in search impressions and clicks, but content teams should combine age with other signals rather than treating the flag as a standalone refresh decision.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.